# 27. Motion Attribution Test (Pipeline 08)

- Goal: verify motion-attribution applicability from exercise definitions, then check the selected real sample outcome.
- Docs: `docs_eng/pipeline/08_motion_attribution.md` / `docs/pipeline/08_motion_attribution.md`
- Prerequisite: 20-26 stage checks should already pass.
- Input: segmented pose dataframe for the selected real sample plus available exercise definitions.
- Output: attribution report/provenance and frame-level attribution columns; coordinates, `rep_id`, and `phase` are not modified.
- Checks: previous-stage setup, definition-driven policy matrix, field readiness coverage, real-sample attribution outcome, synthetic runnable-definition smoke test, output contract, and pipeline integration.
- Policy: this stage supports side-specific feature interpretation only. It does not score movement quality and does not auto-correct labels in conservative mode.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from movement.config import LANDMARKS
from movement.exercise_definition import load_exercise_definition
from movement.motion_attribution import (
    AttributionReport,
    AttributionThresholds,
    attribute_motion,
)
from movement.normalization import check_normalization_result
from movement.pipeline import (
    AnnotationConfig,
    ExerciseDefinitionConfig,
    MotionAttributionConfig,
    NormalizationConfig,
    PhaseSegmentationConfig,
    PipelineConfig,
    PreprocessingConfig,
    RepSegmentationConfig,
    ValidationConfig,
    run_pipeline,
)
from movement.segmentation import segment_phases, segment_reps
from movement.stage_context import prepare_previous_stage_inputs

print('imports OK')

## Data Setup

Runs the previous context in pipeline order before motion attribution:

```text
Pose CSV -> ① Validation -> ② Annotation -> ③ Exercise Definition -> ④ Preprocessing -> ⑤ Normalization -> ⑦ Segmentation
```

⑥ Canonicalization can remain a previous pipeline stage, but active-side attribution uses segmentation context and normalized/preprocessed coordinates rather than corrected candidate coordinates.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Could not find project root containing pyproject.toml')
    PROJECT_ROOT = PROJECT_ROOT.parent

pose_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv'
annotation_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv'
TARGET_EXERCISE_ID = 'draft_squat'

pre_config = PreprocessingConfig(enabled=True)
norm_config = NormalizationConfig(
    enabled=True,
    keep_reference_columns=True,
    model_depth_scale=1.0,
)
stage_inputs = prepare_previous_stage_inputs(
    prepare_until='normalization',
    pose_csv=pose_csv,
    annotation_csv=annotation_csv,
    exercise_id=TARGET_EXERCISE_ID,
    landmarks=LANDMARKS,
    preprocessing_config=pre_config,
    normalization_config=norm_config,
)

df_raw = stage_inputs.raw_df
ann_df = stage_inputs.annotation_df
val_report = stage_inputs.validation_report
ann_report = stage_inputs.annotation_report
exercise_def = stage_inputs.exercise_definition
pre_report = stage_inputs.preprocessing_report
norm_df = stage_inputs.normalized_df
norm_report = stage_inputs.normalization_report
annotation_path = stage_inputs.annotation_csv
TARGET_DEFINITIONS_DIR = stage_inputs.definitions_dir

check_report = check_normalization_result(norm_df)
assert check_report['passed'] is True

df_rep_seg, rep_report = segment_reps(norm_df, exercise_def, fps_default=30.0)
df_seg, phase_reports = segment_phases(df_rep_seg, exercise_def, fps_default=30.0)

setup_summary = pd.DataFrame([
    {'item': 'frames_loaded', 'value': len(df_raw)},
    {'item': 'validation_passed', 'value': val_report['passed']},
    {'item': 'analysis_frames', 'value': f"{ann_report['num_analysis_frames']} / {ann_report['num_total_frames']}"},
    {'item': 'exercise_id', 'value': exercise_def.exercise_id},
    {'item': 'laterality', 'value': exercise_def.classification.get('laterality')},
    {'item': 'execution_pattern', 'value': df_seg['execution_pattern'].dropna().iloc[0] if 'execution_pattern' in df_seg and not df_seg['execution_pattern'].dropna().empty else None},
    {'item': 'side_sequence_mode', 'value': getattr(exercise_def.performance_protocol.side_sequence, 'mode', None) if exercise_def.performance_protocol else None},
    {'item': 'segmented_rep_frames', 'value': int(df_seg['segment_type'].eq('rep').sum())},
    {'item': 'phase_labeled_rep_frames', 'value': int(df_seg.loc[df_seg['segment_type'].eq('rep'), 'phase'].notna().sum())},
    {'item': 'preprocessing_invalid_frames', 'value': pre_report['num_invalid_frames']},
    {'item': 'normalization_scale_value', 'value': round(float(norm_report['scale_value']), 6)},
    {'item': 'definitions_dir', 'value': str(TARGET_DEFINITIONS_DIR)},
])
display(setup_summary)

if val_report.get('warnings'):
    display(pd.DataFrame(val_report['warnings']))

## Definition Policy Matrix

Scan available exercise definitions and classify the motion-attribution policy from definition fields. The table is intentionally definition-driven: adding a new authored exercise should add a row without editing this notebook. For side-specific exercises, `primary_body_regions` and `joint_actions.primary` are treated as readiness gates: they do not change the motion-energy algorithm, but missing context prevents a definition from being marked fully runnable.


In [ ]:
ATTRIBUTION_COLUMNS = [
    'detected_active_limb',
    'expected_active_limb',
    'attribution_consistent',
    'attribution_confidence',
    'attribution_action',
]
POLICY_VALUES = {'skip', 'run', 'conditional', 'not_yet_implemented'}
FIELD_STAGE_USES = {'uses', 'carries', 'downstream', 'not_yet_supported'}


def definition_files(project_root):
    roots = [
        ('runtime', project_root / 'data' / 'definitions' / 'exercises'),
        ('local_authoring_draft', project_root / 'data' / 'processed' / 'authoring_drafts'),
        ('example_authoring_bundle', project_root / 'data' / 'examples' / 'exercise_authoring'),
    ]
    for source_kind, root in roots:
        if not root.exists():
            continue
        if source_kind == 'runtime':
            yield from ((source_kind, path) for path in sorted(root.glob('*.yaml')))
        else:
            pattern = '*/data/definitions/exercises/*.yaml'
            yield from ((source_kind, path) for path in sorted(root.glob(pattern)))


def load_raw_definition(yaml_path):
    with yaml_path.open(encoding='utf-8') as fh:
        return yaml.safe_load(fh) or {}


def compact_value(value):
    if value is None:
        return None
    if isinstance(value, (list, tuple)):
        return ', '.join(str(item) for item in value) if value else '[]'
    if isinstance(value, dict):
        return ', '.join(f'{k}={v}' for k, v in value.items()) if value else '{}'
    return value


def pairable_primary_pairs(definition):
    primary = list(definition.landmarks.primary_joints or [])
    primary_set = set(primary)
    pairs = []
    for landmark in primary:
        if landmark.startswith('left_'):
            right = 'right_' + landmark[5:]
            if right in primary_set:
                pairs.append((landmark, right))
    return pairs


def side_sequence_mode(definition):
    protocol = getattr(definition, 'performance_protocol', None)
    if protocol is None:
        return None
    return getattr(protocol.side_sequence, 'mode', None)


def primary_body_regions(raw):
    return (
        raw.get('primary_body_regions')
        or (raw.get('biomechanical_identity') or {}).get('primary_body_regions')
        or (raw.get('authoring_spec') or {}).get('primary_body_regions')
        or []
    )


def primary_joint_actions(raw):
    joint_actions = raw.get('joint_actions') or {}
    biomechanical_identity = raw.get('biomechanical_identity') or {}
    authoring_spec = raw.get('authoring_spec') or {}
    return (
        joint_actions.get('primary')
        or biomechanical_identity.get('primary_joint_actions')
        or authoring_spec.get('primary_joint_actions')
        or []
    )


def attribution_readiness_missing(definition, raw):
    missing = []
    if not pairable_primary_pairs(definition):
        missing.append('landmarks.primary_joints.left_right_pairs')
    if not primary_body_regions(raw):
        missing.append('primary_body_regions')
    if not primary_joint_actions(raw):
        missing.append('joint_actions.primary')
    return missing


def motion_attribution_policy(definition, raw=None):
    raw = raw or {}
    laterality = definition.classification.get('laterality')
    pairs = pairable_primary_pairs(definition)
    mode = side_sequence_mode(definition)
    missing = attribution_readiness_missing(definition, raw)

    if laterality == 'bilateral_symmetric':
        return 'skip', 'bilateral symmetric exercise has no active-side concept', missing
    if laterality == 'bilateral_asymmetric':
        return 'not_yet_implemented', 'route to side-bias/symmetry features until active-side semantics are defined', missing
    if laterality in {'alternating', 'unilateral_left', 'unilateral_right'}:
        if missing:
            return 'conditional', 'requires definition readiness fields: ' + ', '.join(missing), missing
        return 'run', f'active-side attribution can use {len(pairs)} pairable primary-joint pairs', missing
    if laterality == 'unilateral_unspecified':
        if missing:
            return 'conditional', 'requires definition readiness fields: ' + ', '.join(missing), missing
        if mode not in (None, 'none'):
            return 'conditional', 'requires starting-side/context evidence before active-side attribution', missing
        return 'conditional', 'requires context/evidence to infer the expected side', missing
    return 'skip', f'laterality={laterality!r} is not applicable to active-side attribution', missing


def definition_field_rows(source_kind, yaml_path, definition, raw, policy):
    classification = definition.classification
    protocol = getattr(definition, 'performance_protocol', None)
    side_sequence = getattr(protocol, 'side_sequence', None) if protocol else None
    prescription = getattr(protocol, 'prescription', None) if protocol else None
    support = raw.get('support') or {}
    authoring_spec = raw.get('authoring_spec') or {}
    joint_actions = raw.get('joint_actions') or {}
    phase_seg = getattr(definition, 'phase_segmentation', None)
    camera_protocol = getattr(definition, 'camera_protocol', None)

    rows = [
        ('classification.laterality', classification.get('laterality'), 'uses', 'primary gate for skip/run/conditional attribution policy'),
        ('landmarks.primary_joints', definition.landmarks.primary_joints, 'uses', 'left/right pair availability for active-side evidence'),
        ('primary_body_regions', primary_body_regions(raw), 'uses', 'readiness gate for interpreting active-side evidence'),
        ('joint_actions.primary', primary_joint_actions(raw), 'uses', 'readiness gate for interpreting active-side evidence'),
        ('performance_protocol.side_sequence.mode', getattr(side_sequence, 'mode', None), 'uses', 'expected active-side order when a side-specific path is applicable'),
        ('performance_protocol.side_sequence.block_size_counts', getattr(side_sequence, 'block_size_counts', None), 'uses', 'same-side block size when side sequence mode requires it'),
        ('classification.movement_template_id', classification.get('movement_template_id'), 'carries', 'context label; this stage does not branch by a named exercise template'),
        ('classification.movement_pattern', classification.get('movement_pattern'), 'carries', 'derived descriptor, not a hardcoded algorithm selector'),
        ('classification.posture_type', classification.get('posture_type'), 'carries', 'context for upstream normalization/canonicalization and downstream feature interpretation'),
        ('classification.body_geometry', classification.get('body_geometry'), 'carries', 'context for posture-aware stages; no active-side branch yet'),
        ('classification.primary_plane', classification.get('primary_plane'), 'carries', 'plane context for feature interpretation; no active-side branch yet'),
        ('classification.secondary_planes', classification.get('secondary_planes'), 'carries', 'plane context for feature interpretation; no active-side branch yet'),
        ('support.base_of_support', support.get('base_of_support'), 'downstream', 'support context is used by normalization/canonicalization and later feature policy, not active-side attribution'),
        ('support.contact_points', support.get('contact_points'), 'downstream', 'contact context is preserved for support-aware stages'),
        ('joint_actions.secondary', joint_actions.get('secondary'), 'carries', 'secondary joint-action semantics preserved for feature selection and interpretation'),
        ('phase_segmentation.phase_sequence', getattr(phase_seg, 'phase_sequence', None), 'downstream', 'phase labels must be preserved; ⑧ does not relabel phases'),
        ('performance_protocol.prescription.target_count_per_set', getattr(prescription, 'target_count_per_set', None), 'carries', 'performance prescription is provenance here; counting quality is not scored in ⑧'),
        ('camera_protocol', camera_protocol, 'downstream', 'view context is retained for reliability policy, not for active-side detection'),
        ('authoring_spec.phase_template', authoring_spec.get('phase_template'), 'carries', 'authoring provenance retained for generated definitions'),
        ('authoring_spec.counting_template', authoring_spec.get('counting_template'), 'carries', 'authoring provenance retained for generated definitions'),
        ('authoring_spec.analysis_template', authoring_spec.get('analysis_template'), 'carries', 'authoring provenance retained for generated definitions'),
    ]
    if classification.get('laterality') == 'bilateral_asymmetric' or policy == 'not_yet_implemented':
        rows.append((
            'classification.laterality.bilateral_asymmetric',
            classification.get('laterality'),
            'not_yet_supported',
            'needs a side-bias/symmetry feature policy instead of active-side attribution',
        ))

    return [
        {
            'source_kind': source_kind,
            'exercise_id': definition.exercise_id,
            'field': field,
            'value': compact_value(value),
            'stage_use': stage_use,
            'note': note,
            'definitions_dir': str(yaml_path.parent),
        }
        for field, value, stage_use, note in rows
    ]


policy_rows = []
field_rows = []
loaded_definitions = []
seen = set()
for source_kind, yaml_path in definition_files(PROJECT_ROOT):
    key = (source_kind, str(yaml_path.parent.resolve()), yaml_path.stem)
    if key in seen:
        continue
    seen.add(key)
    raw_definition = load_raw_definition(yaml_path)
    try:
        definition = load_exercise_definition(
            exercise_id=yaml_path.stem,
            definitions_dir=yaml_path.parent,
        )
        policy, reason, missing_readiness = motion_attribution_policy(definition, raw_definition)
        pairs = pairable_primary_pairs(definition)
        loaded_definitions.append((source_kind, yaml_path, definition, policy))
        policy_rows.append({
            'source_kind': source_kind,
            'exercise_id': definition.exercise_id,
            'display_name': definition.display_name,
            'laterality': definition.classification.get('laterality'),
            'movement_template_id': definition.classification.get('movement_template_id'),
            'side_sequence_mode': side_sequence_mode(definition),
            'primary_pair_count': len(pairs),
            'readiness_missing': ', '.join(missing_readiness) if missing_readiness else None,
            'policy': policy,
            'reason': reason,
            'definitions_dir': str(yaml_path.parent),
        })
        field_rows.extend(definition_field_rows(source_kind, yaml_path, definition, raw_definition, policy))
    except Exception as exc:
        policy_rows.append({
            'source_kind': source_kind,
            'exercise_id': yaml_path.stem,
            'display_name': None,
            'laterality': None,
            'movement_template_id': None,
            'side_sequence_mode': None,
            'primary_pair_count': 0,
            'readiness_missing': None,
            'policy': 'conditional',
            'reason': f'load failed: {exc}',
            'definitions_dir': str(yaml_path.parent),
        })

policy_df = pd.DataFrame(policy_rows).sort_values(['source_kind', 'exercise_id']).reset_index(drop=True)
field_coverage_df = pd.DataFrame(field_rows).sort_values(['source_kind', 'exercise_id', 'field']).reset_index(drop=True)

display(policy_df[['source_kind', 'exercise_id', 'laterality', 'side_sequence_mode', 'primary_pair_count', 'readiness_missing', 'policy', 'reason']])

target_field_coverage = field_coverage_df[field_coverage_df['exercise_id'].eq(TARGET_EXERCISE_ID)]
display(target_field_coverage[['source_kind', 'field', 'value', 'stage_use', 'note']])


## Check 1: Policy Matrix Covers Applicable And Non-Applicable Cases

This check is about extension readiness. It should show skipped bilateral definitions, runnable alternating/unilateral definitions when present, and explicit conditional/not-yet-implemented rows for definition fields that are not fully supported yet. It also checks that the selected definition has visible field coverage, including the readiness fields needed to interpret active-side evidence.


In [ ]:
assert not policy_df.empty, 'no exercise definitions discovered'
assert set(policy_df['policy']).issubset(POLICY_VALUES)
assert TARGET_EXERCISE_ID in set(policy_df['exercise_id'])
assert not field_coverage_df.empty, 'definition field coverage was not generated'
assert set(field_coverage_df['stage_use']).issubset(FIELD_STAGE_USES)
assert {'uses', 'carries'}.issubset(set(target_field_coverage['stage_use']))

run_ready = policy_df[policy_df['policy'].eq('run')]
assert run_ready['readiness_missing'].isna().all(), 'runnable definitions should not be missing readiness fields'

policy_counts = policy_df['policy'].value_counts().rename('definitions').to_frame()
field_use_counts = target_field_coverage['stage_use'].value_counts().rename('target_fields').to_frame()
display(policy_counts)
display(field_use_counts)

real_policy, real_policy_reason, real_missing_readiness = motion_attribution_policy(exercise_def, load_raw_definition(TARGET_DEFINITIONS_DIR / f'{TARGET_EXERCISE_ID}.yaml'))
assert real_policy == 'skip'
assert exercise_def.classification.get('laterality') == 'bilateral_symmetric'
print(f'PASS: target real sample policy = {real_policy} ({real_policy_reason})')


## Direct Real-Sample Motion Attribution Test

Run `attribute_motion()` on the selected real sample. For `draft_squat`, the expected outcome is a skipped stage with complete provenance because the movement is bilateral symmetric.

In [ ]:
thresholds = AttributionThresholds(active=0.70, ambiguous=0.55, swap=0.85)
df_attr, attr_report = attribute_motion(
    df=df_seg,
    exercise_definition=exercise_def,
    thresholds=thresholds,
    mode='conservative',
)
attr_dict = attr_report.as_dict()

real_attr_summary = pd.DataFrame([
    {'item': 'exercise_id', 'value': attr_dict['exercise_id']},
    {'item': 'laterality', 'value': attr_dict['laterality']},
    {'item': 'execution_pattern', 'value': attr_dict['execution_pattern']},
    {'item': 'side_sequence', 'value': attr_dict['performance_side_sequence']},
    {'item': 'skipped', 'value': attr_dict['skipped']},
    {'item': 'skip_reason', 'value': attr_dict['skip_reason']},
    {'item': 'landmark_pairs_used', 'value': len(attr_dict['landmark_pairs_used'])},
])
display(real_attr_summary)

## Check 2: Real-Sample Outcome Matches Definition Policy

In [ ]:
assert isinstance(attr_report, AttributionReport)
assert attr_dict['exercise_id'] == TARGET_EXERCISE_ID
assert attr_dict['laterality'] == exercise_def.classification.get('laterality')
assert attr_dict['execution_pattern'] == 'bilateral'
assert attr_dict['performance_side_sequence']['mode'] == 'none'
assert attr_dict['thresholds']
assert attr_dict['landmark_pairs_used']

if real_policy == 'skip':
    assert attr_dict['skipped'] is True
    assert attr_dict['skip_reason']
else:
    assert attr_dict['skipped'] is False
print('PASS: real-sample attribution outcome matches the definition-driven policy')

## Check 3: Output Contract And Non-Mutation

Motion attribution may add metadata columns. It must not alter coordinates, `rep_id`, or `phase`. For a skipped bilateral sample, attribution values should remain null.

In [ ]:
for col in ATTRIBUTION_COLUMNS:
    assert col in df_attr.columns, f'missing attribution column: {col}'

if attr_dict['skipped']:
    null_counts = {col: int(df_attr[col].notna().sum()) for col in ATTRIBUTION_COLUMNS}
    display(pd.Series(null_counts, name='non_null_frames').to_frame())
    assert all(count == 0 for count in null_counts.values())

coord_cols = [c for c in df_seg.columns if c.endswith(('_norm_x', '_norm_y', '_norm_z'))]
for col in coord_cols:
    assert np.allclose(df_attr[col].to_numpy(), df_seg[col].to_numpy(), equal_nan=True), f'{col} was modified'

pd.testing.assert_series_equal(df_attr['rep_id'], df_seg['rep_id'], check_names=False)
pd.testing.assert_series_equal(df_attr['phase'], df_seg['phase'], check_names=False)
print(f'PASS: attribution columns present; coordinates/rep_id/phase unchanged ({len(coord_cols)} norm coordinate columns checked)')

## Synthetic Applicability Smoke Test

When at least one available definition has `policy == run`, build a tiny generated dataframe from that definition and confirm the active-side path runs. This is not a movement-quality evaluation and does not replace real recordings.

In [ ]:
def other_side(side):
    return 'left' if side == 'right' else 'right'


def synthetic_sequence_for(definition, starting_side='right'):
    laterality = definition.classification.get('laterality')
    protocol = getattr(definition, 'performance_protocol', None)
    side_sequence = getattr(protocol, 'side_sequence', None) if protocol else None
    mode = getattr(side_sequence, 'mode', 'none') if side_sequence else 'none'

    if laterality == 'unilateral_left':
        return ['left'] * 2, 'bilateral', 'left'
    if laterality == 'unilateral_right':
        return ['right'] * 2, 'bilateral', 'right'
    if mode == 'same_side_block_then_switch':
        block = int(getattr(side_sequence, 'block_size_counts', None) or 2)
        return [starting_side] * block + [other_side(starting_side)] * block, 'bilateral', starting_side
    if mode == 'alternating_each_rep' or laterality == 'alternating':
        return [starting_side, other_side(starting_side), starting_side, other_side(starting_side)], 'alternating', starting_side
    return [], 'bilateral', starting_side


def build_synthetic_attribution_df(definition, active_sequence, execution_pattern, starting_side):
    pairs = pairable_primary_pairs(definition)
    if not pairs:
        raise ValueError('synthetic attribution requires pairable primary joints')
    rows = []
    frame = 0
    frames_per_rep = 5
    for rep_index, active_side in enumerate(active_sequence, start=1):
        for local_frame in range(frames_per_rep):
            row = {
                'frame': frame,
                'segment_type': 'rep',
                'rep_id': rep_index,
                'phase': 'Synthetic',
                'exercise_id': definition.exercise_id,
                'execution_pattern': execution_pattern,
                'starting_side': starting_side,
            }
            for left, right in pairs:
                for landmark, amplitude in ((left, 0.1), (right, 0.1)):
                    if landmark.startswith(active_side + '_'):
                        amplitude = 1.0
                    row[f'{landmark}_norm_x'] = amplitude * float(local_frame)
                    row[f'{landmark}_norm_y'] = 0.0
                    row[f'{landmark}_norm_z'] = 0.0
            rows.append(row)
            frame += 1
    return pd.DataFrame(rows)

run_candidates = [
    (source_kind, yaml_path, definition)
    for source_kind, yaml_path, definition, policy in loaded_definitions
    if policy == 'run'
]

if run_candidates:
    source_kind, yaml_path, synthetic_definition = sorted(
        run_candidates,
        key=lambda item: (item[0] != 'runtime', item[2].exercise_id),
    )[0]
    active_sequence, synthetic_execution_pattern, synthetic_starting_side = synthetic_sequence_for(
        synthetic_definition
    )
    synthetic_df = build_synthetic_attribution_df(
        synthetic_definition,
        active_sequence,
        synthetic_execution_pattern,
        synthetic_starting_side,
    )
    synthetic_attr_df, synthetic_report = attribute_motion(
        synthetic_df,
        synthetic_definition,
        thresholds=thresholds,
        mode='conservative',
    )
    synthetic_summary = pd.DataFrame([
        {'item': 'source_kind', 'value': source_kind},
        {'item': 'exercise_id', 'value': synthetic_definition.exercise_id},
        {'item': 'laterality', 'value': synthetic_definition.classification.get('laterality')},
        {'item': 'side_sequence_mode', 'value': side_sequence_mode(synthetic_definition)},
        {'item': 'synthetic_active_sequence', 'value': active_sequence},
        {'item': 'skipped', 'value': synthetic_report.skipped},
        {'item': 'num_reps', 'value': synthetic_report.num_reps},
        {'item': 'num_consistent', 'value': synthetic_report.num_consistent},
        {'item': 'num_flagged', 'value': synthetic_report.num_flagged},
    ])
    display(synthetic_summary)
else:
    synthetic_definition = None
    synthetic_attr_df = pd.DataFrame()
    synthetic_report = None
    print('No runnable definitions found for synthetic smoke test.')

## Check 4: Runnable Definition Path Works When Available

In [ ]:
if synthetic_report is None:
    print('SKIP: no definition with policy == run is currently available.')
else:
    assert synthetic_report.skipped is False
    assert synthetic_report.num_reps == len(active_sequence)
    assert synthetic_report.num_consistent == len(active_sequence)
    per_rep_expected = [
        group['expected_active_limb'].dropna().iloc[0]
        for _, group in synthetic_attr_df.groupby('rep_id', sort=True)
    ]
    per_rep_detected = [
        group['detected_active_limb'].dropna().iloc[0]
        for _, group in synthetic_attr_df.groupby('rep_id', sort=True)
    ]
    assert per_rep_expected == active_sequence
    assert per_rep_detected == active_sequence
    print(f'PASS: synthetic active-side path ran for {synthetic_report.exercise_id}')

## Check 5: Pipeline Integration

Run the selected real sample through `run_pipeline()` and verify that the motion-attribution report and output contract match the direct call.

In [ ]:
cfg = PipelineConfig()
cfg.validation = ValidationConfig(enabled=True)
cfg.annotation = AnnotationConfig(enabled=True, path=annotation_path)
cfg.exercise_definition = ExerciseDefinitionConfig(
    enabled=True,
    definitions_dir=str(TARGET_DEFINITIONS_DIR),
    exercise_id=TARGET_EXERCISE_ID,
)
cfg.preprocessing = pre_config
cfg.normalization = norm_config
cfg.canonicalization.enabled = False
cfg.rep_segmentation = RepSegmentationConfig(enabled=True, fps_default=30.0)
cfg.phase_segmentation = PhaseSegmentationConfig(enabled=True, fps_default=30.0)
cfg.motion_attribution = MotionAttributionConfig(enabled=True)

pipe_df, pipe_report = run_pipeline(
    df_raw,
    config=cfg,
    landmarks=LANDMARKS,
    ann_df=ann_df,
)

assert 'motion_attribution' in pipe_report
pipe_attr_report = pipe_report['motion_attribution']
assert pipe_attr_report['exercise_id'] == TARGET_EXERCISE_ID
assert pipe_attr_report['laterality'] == attr_dict['laterality']
assert pipe_attr_report['skipped'] == attr_dict['skipped']
for col in ATTRIBUTION_COLUMNS:
    assert col in pipe_df.columns

rep_mask_pipe = pipe_df['segment_type'] == 'rep'
print(f"pipeline ⑧ skipped: {pipe_attr_report['skipped']} ({pipe_attr_report['skip_reason']})")
print(f'pipeline rep frames: {int(rep_mask_pipe.sum())}')
print(f'steps executed: {list(pipe_report.keys())}')
print('PASS: pipeline integration preserves the direct motion-attribution policy outcome')

## Check Summary

This notebook is a compact execution/QC checkpoint for ⑧ Motion Attribution. It verifies the selected real sample, records how exercise-definition fields are used or carried forward, checks readiness fields for side-specific attribution, and confirms definition-driven extension readiness for non-bilateral exercises without requiring additional recordings.
